# The annual friends' weekend — stage 4 chart (Analysis 7)

Builds the primary + companion charts designed in `analysis-log.md`, Analysis 7,
stage 4, for the proposition:

> Message volume is higher than the group's normal baseline in the 60 days before
> the weekend and during the weekend itself, and stays elevated for about a week
> after, before returning to baseline.

Loads the already-anonymised, already-featured export from
[`01.3-your-own-chat.ipynb`](../lesson1/01.3-your-own-chat.ipynb) (same
`config.toml` `current` file) — no re-parsing here, this notebook only adds the
weekend-window features and the stage-4 charts on top.

In [ ]:
import pandas as pd

from goad_toolkit.datatransforms import FlagDates, GroupAgg, Pipeline
from goad_toolkit.visualizer import Annotate, FacetPlot, LinePlot, PlotSettings, VerticalLine

from wa_analyzer.data import load_own_chat

own = load_own_chat()
own.shape

## Reference dates

Copied from `analysis-log.md` stage 1/2 — the two weekends the whole group attended.
`BEFORE_DAYS` / `AFTER_DAYS` are stage 2's window widths (60 days before, 7 days
after), confirmed with the student after an initial units mixup ("60 month" ->
60 days).

In [ ]:
WEEKENDS = {
    "2024": (pd.Timestamp("2024-04-19"), pd.Timestamp("2024-04-21")),
    "2025": (pd.Timestamp("2025-09-19"), pd.Timestamp("2025-09-21")),
}
BEFORE_DAYS = 30  # widened from 60 -> 30 at the student's request, a direct sensitivity
AFTER_DAYS = 30   # check on the stage-4 parameter flag: does the read change with a
                  # narrower before-window and a much wider after-window?

# Stage 3's birthday-collision check, as calendar dates -- fed into FlagDates below.
# Re-checked against the widened windows: hypnotic-rabbit's birthday (Oct 19) now
# falls inside 2025's 30-day after-window (previously outside the 7-day one).
BIRTHDAY_COLLISIONS = {
    "2024-04-13": "fluffy-beaver's birthday",
    "2025-09-16": "effervescent-penguin's birthday",
    "2025-09-22": "striking-rail's birthday",
    "2025-10-19": "hypnotic-rabbit's birthday",
}

## Daily counts, reindexed to a full calendar

`GroupAgg` only returns days that have at least one message. A day with zero
messages is real information here -- the pre-trip ramp-up (or its absence) needs
the quiet days visible, not dropped -- so the daily series is reindexed to every
calendar day in the export, filling gaps with 0.

In [ ]:
own["date"] = pd.to_datetime(own["timestamp"]).dt.tz_localize(None).dt.normalize()

full_range = pd.date_range(own["date"].min(), own["date"].max(), freq="D")
counted = Pipeline().add(GroupAgg, by="date", agg="size", feature="messages").apply(own)
daily = (
    counted.set_index("date")["messages"]
    .reindex(full_range, fill_value=0)
    .rename_axis("date")
    .reset_index()
)

daily = Pipeline().add(
    FlagDates, column="date", dates=list(BIRTHDAY_COLLISIONS), feature="is_birthday_collision"
).apply(daily)

daily.shape

## Event-aligned window: `days_relative`, `weekend_id`

Stage 2's core features. Each weekend gets its own before/during/after slice of the
daily series (`start - BEFORE_DAYS` to `end + AFTER_DAYS`), re-expressed as an
offset from that weekend's own start date -- `days_relative = 0` is the first day
of the trip for both years, so 2024 and 2025 can be overlaid on the same axis
despite being five months apart on the calendar.

In [ ]:
frames = []
for weekend_id, (start, end) in WEEKENDS.items():
    lo = start - pd.Timedelta(days=BEFORE_DAYS)
    hi = end + pd.Timedelta(days=AFTER_DAYS)
    sub = daily[(daily["date"] >= lo) & (daily["date"] <= hi)].copy()
    sub["weekend_id"] = weekend_id
    sub["days_relative"] = (sub["date"] - start).dt.days
    frames.append(sub)

event_days = pd.concat(frames, ignore_index=True)
event_days.groupby("weekend_id")["messages"].describe()

## `period` bucket, for the secondary (supporting) chart

Stage 3/4's decision: the time series above is the primary read; this
before/during/after/baseline bucketing is a secondary, supporting view --
`during` only covers 6 days total across both events, too thin to carry equal
weight as its own bucket in the headline chart. `baseline` excludes every
weekend's own before/during/after window, so one weekend's halo doesn't bias the
other's baseline.

In [ ]:
period = pd.Series("baseline", index=daily.index)
for weekend_id, (start, end) in WEEKENDS.items():
    before_lo, before_hi = start - pd.Timedelta(days=BEFORE_DAYS), start - pd.Timedelta(days=1)
    after_lo, after_hi = end + pd.Timedelta(days=1), end + pd.Timedelta(days=AFTER_DAYS)
    period[(daily["date"] >= before_lo) & (daily["date"] <= before_hi)] = "before"
    period[(daily["date"] >= start) & (daily["date"] <= end)] = "during"
    period[(daily["date"] >= after_lo) & (daily["date"] <= after_hi)] = "after"
daily["period"] = period.values

period_summary = daily.groupby("period")["messages"].agg(["count", "median", "mean"])
period_summary.reindex(["baseline", "before", "during", "after"])

## Stage 4's new confound: is the pre-weekend rise just "Friday chatter"?

Surfaced by the student at stage 4: the `before`-window's own composition (many
Fridays/weekends inside a 60-day span) could produce a rise in raw counts that has
nothing to do with *this* trip -- it could just be the group's normal weekly
rhythm. Compares each day-of-week's median across the whole chat against the same
day-of-week's median inside the `before` window only.

In [ ]:
daily["day_of_week"] = daily["date"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

dow_check = pd.DataFrame({
    "overall_median": daily.groupby("day_of_week")["messages"].median(),
    "before_window_median": daily[daily["period"] == "before"].groupby("day_of_week")["messages"].median(),
}).reindex(dow_order)
dow_check

## Primary chart: event-aligned time series, both years overlaid

The single comparison this plot exists to make (stage 4): does chat activity look
visibly different during the weekend-away period compared to normal -- read
directly off the line, not off a single summary number. Vertical lines mark the
trip's start (`days_relative = 0`) and end (`days_relative = 2`, a 3-day weekend
for both years). The three stage-3 birthday collisions are annotated directly, so
a reader can see which bumps are a birthday and which aren't.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5.8))

# Raw daily counts, one shade family per year (kept per stage 3's "label spikes,
# don't smooth them away" -- thin/faded rather than deleted). Each year's raw
# line and its 3-day rolling-mean line share a hue family (light grey vs.
# near-black) so the two are distinguishable without a legend.
YEAR_STYLE = {
    "2024": {"raw": "#cccccc", "smooth": "#999999"},
    "2025": {"raw": "#666666", "smooth": "#222222"},
}
smoothed_ends = {}
for weekend_id, style in YEAR_STYLE.items():
    sub = event_days[event_days["weekend_id"] == weekend_id].sort_values("days_relative")
    ax.plot(sub["days_relative"], sub["messages"], color=style["raw"], linewidth=0.8, alpha=0.8)
    smoothed = sub["messages"].rolling(3, center=True, min_periods=1).mean()
    ax.plot(sub["days_relative"], smoothed, color=style["smooth"], linewidth=2.5)
    smoothed_ends[weekend_id] = (sub["days_relative"].iloc[-1], smoothed.iloc[-1])

# During-window as a shaded span -- the one comparison the plot exists to make.
ax.axvspan(-0.5, 2.5, color="crimson", alpha=0.15)

# Baseline reference, so "elevated" has something concrete to be judged against.
# Label placed above the busy cluster of lines near y=0-20, with a short leader
# down to the actual dotted line -- the first version sat right on top of the
# lines it was meant to be read against.
baseline_median = period_summary.loc["baseline", "median"]
ax.axhline(baseline_median, color="black", linestyle=":", linewidth=1)
ax.annotate(
    f"Baseline median ({baseline_median:.0f}/day)",
    xy=(-27, baseline_median), xytext=(-27, 30),
    fontsize=9, color="black", ha="center",
    arrowprops=dict(arrowstyle="-", color="black", lw=0.8),
)

# Birthday collisions -- a small muted marker, not a star, so it registers as a
# footnote-level detail rather than competing with the main trend.
first_birthday_xy = None
for date_str in BIRTHDAY_COLLISIONS:
    row = event_days[event_days["date"] == pd.Timestamp(date_str)]
    if row.empty:
        continue
    x, y = row["days_relative"].iloc[0], row["messages"].iloc[0]
    ax.scatter(
        [x], [y], marker="o", s=45, facecolor="none",
        edgecolor="#888888", linewidth=1.3, zorder=5,
    )
    if first_birthday_xy is None:
        first_birthday_xy = (x, y)

ax.set_xticks(np.arange(-30, 31, 5))
ax.set_xlabel("days from the weekend's start (0 = day 1 of the trip)")
ax.set_ylabel("messages per day")

# Direct in-plot labels instead of a legend box -- each element labelled where
# it sits, so the eye never has to jump to a key on the side.
ax.text(
    smoothed_ends["2024"][0] + 0.6, smoothed_ends["2024"][1], "2024",
    color=YEAR_STYLE["2024"]["smooth"], fontsize=10, fontweight="bold", va="center",
)
ax.text(
    smoothed_ends["2025"][0] + 0.6, smoothed_ends["2025"][1], "2025",
    color=YEAR_STYLE["2025"]["smooth"], fontsize=10, fontweight="bold", va="center",
)
ax.text(
    1, ax.get_ylim()[1] * 0.97, "During the\nweekend",
    color="crimson", fontsize=9, ha="center", va="top",
)
if first_birthday_xy is not None:
    ax.annotate(
        "birthday", xy=first_birthday_xy,
        xytext=(first_birthday_xy[0] - 3, first_birthday_xy[1] + 14),
        fontsize=8, color="#888888",
        arrowprops=dict(arrowstyle="-", color="#888888", lw=0.8),
    )

# Headline/subtitle split (same pattern as the election-length slide charts):
# a challenging headline as the hook, the precise measured claim demoted to an
# italic subtitle underneath.
SUBTITLE = (
    "Sharp rise during and right after the weekend, fading within 1-2 weeks -- "
    "no comparable rise in the 30 days before"
)
ax.set_title(SUBTITLE, fontsize=10, style="italic", pad=8)
fig.suptitle(
    "We don't anticipate the friends' weekend trip -- we react to it",
    fontsize=15, fontweight="bold", y=0.98,
)

# Footnote: shorter than the first draft, and now covers both shuffle tests --
# during (essentially unique) and after (real, but not rare) read very
# differently, and that contrast matters for how much weight either deserves.
fig.text(
    0.06, 0.02,
    "* Shuffle test vs. ~2090 comparable baseline windows elsewhere in this chat's history:\n"
    "the during spike (80/day) is unmatched (0 as high) -- the after level (20.5/day) is real but not rare (68 windows, ~3%, reach it too).",
    ha="left", va="bottom", fontsize=7.5, style="italic", color="#444444",
)

# subplots_adjust instead of tight_layout(rect=...) -- the rect version left a
# large blank gap between the subtitle and the plot that tight_layout's own
# heuristics kept re-inserting.
fig.subplots_adjust(top=0.78, bottom=0.22, left=0.07, right=0.90)
fig.savefig("friends-weekend-timeseries.png", dpi=200)

## Secondary chart: before/during/after/baseline, median messages per day

Supporting view, not the headline (stage 3/4 decision) -- one bar per period,
median rather than mean per stage 3's spike-robustness decision. `during` is
labelled with its thin sample size (n=6 days) directly, so a reader isn't left to
assume it rests on the same footing as the other three bars.

In [ ]:
order = ["baseline", "before", "during", "after"]
plot_data = period_summary.reindex(order).reset_index()
plot_data["label"] = [f"{p}\n(n={c} days)" for p, c in zip(plot_data["period"], plot_data["count"])]

bar_settings = PlotSettings(
    figsize=(8, 5),
    title="Median messages/day by period",
    xlabel="",
    ylabel="median messages per day",
)
fig, ax = __import__("matplotlib.pyplot", fromlist=["subplots"]).subplots(figsize=bar_settings.figsize)
ax.bar(plot_data["label"], plot_data["median"], color="#333333")
ax.set_title(bar_settings.title)
ax.set_ylabel(bar_settings.ylabel)
fig.savefig("friends-weekend-period-medians.png", dpi=200, bbox_inches="tight")

## Companion chart: same event-aligned series, split per author

Stage 2's agreed robustness check -- confirms any before/during/after pattern
holds across all 9 people rather than being driven by 1-2, same pattern as
Analysis 1's per-author companion chart.

In [ ]:
own_authored = own.copy()
frames_author = []
for weekend_id, (start, end) in WEEKENDS.items():
    lo = start - pd.Timedelta(days=BEFORE_DAYS)
    hi = end + pd.Timedelta(days=AFTER_DAYS)
    sub = own_authored[(own_authored["date"] >= lo) & (own_authored["date"] <= hi)].copy()
    sub["weekend_id"] = weekend_id
    sub["days_relative"] = (sub["date"] - start).dt.days
    frames_author.append(sub)
event_by_author = pd.concat(frames_author, ignore_index=True)

per_author_daily = Pipeline().add(
    GroupAgg, by=["author", "weekend_id", "days_relative"], agg="size", feature="messages"
).apply(event_by_author)

facet_settings = PlotSettings(
    figsize=(14, 10),
    title="Messages per day per author, aligned on the weekend",
    ylabel="messages per day",
    sharey=True,
    max_cols=3,
    legend_title="Year",
)
facet = FacetPlot(facet_settings)
fig, axes = facet.plot(
    inner=LinePlot(facet_settings), data=per_author_daily, by="author",
    x="days_relative", y="messages", hue="weekend_id",
)
for ax in axes:
    ax.axvline(-0.5, color="crimson", linestyle="--", linewidth=1)
    ax.axvline(2.5, color="crimson", linestyle="--", linewidth=1)
fig.savefig("friends-weekend-per-author.png", dpi=200, bbox_inches="tight")

## Next: stage 5 (critique)

Look at the primary chart above -- away and back, the way
`goad_analysis_checklist`'s stage-5 table asks -- before the next chat message
answers its questions. That interview needs your own read of the picture, not a
description of what the code above was trying to do.